# Simulating the Amplitude-Based Collisionless QLBM

In [ ]:
from qiskit_aer import AerSimulator

from qlbm.components import (
    ABGridMeasurement,
    EmptyPrimitive,
)
from qlbm.infra import QiskitRunner, SimulationConfig
from qlbm.lattice import ABLattice
from qlbm.tools.utils import create_directory_and_parents

In [ ]:
lattice = ABLattice(
    {
        "lattice": {"dim": {"x": 16, "y": 16}, "velocities": "d2q9"},
        "geometry": [],
    }
)

output_dir = "qlbm-output/ab-pbc-d2q9-16x16-qiskit"
create_directory_and_parents(output_dir)

In [ ]:
# Geometry of marker |0>: two "stacked" "wide" rectangles
# Geometry of the marker |1>: only the "top" rectangle is present
# These geometries will act fully in parallel, and we will observe the average behavior
lattice.set_geometries(
    [
        [
            {"shape": "cuboid", "x": [6, 12], "y": [12, 14], "boundary": "bounceback"},
            {"shape": "cuboid", "x": [6, 12], "y": [8, 10], "boundary": "bounceback"},
        ],
        [{"shape": "cuboid", "x": [6, 12], "y": [12, 14], "boundary": "bounceback"},],
    ]
)

In [ ]:
lattice.geometries

In [ ]:
from qlbm.components.ab.ab import ABQLBM
from qlbm.components.ab.initial import ABParallelDiscreteUniformInitialConditions

cfg = SimulationConfig(
    initial_conditions=ABParallelDiscreteUniformInitialConditions(
        lattice,
        [[5], [5]],
        [([0, 1], [0, 1, 2, 3]), ([0, 1], [0, 1, 2, 3])],
    ),
    algorithm=ABQLBM(lattice, use_agnostic_bcs=False),
    postprocessing=EmptyPrimitive(lattice),
    measurement=ABGridMeasurement(lattice),
    target_platform="QISKIT",
    compiler_platform="QISKIT",
    optimization_level=0,
    statevector_sampling=True,
    execution_backend=AerSimulator(method="statevector"),
    sampling_backend=AerSimulator(method="statevector"),
)

In [ ]:
cfg.prepare_for_simulation()

In [ ]:
# Number of shots to simulate for each timestep when running the circuit
NUM_SHOTS = 2**12

# Number of timesteps to simulate
NUM_STEPS = 20

In [ ]:
runner = QiskitRunner(
    cfg,
    lattice,
)


# Simulate the circuits using both snapshots
runner.run(
    NUM_STEPS,  # Number of time steps
    NUM_SHOTS,  # Number of shots per time step
    output_dir,
    statevector_snapshots=True,
)